# Lab 1 — PCA y monitoreo estructural

Aplicamos **PCA** (reducción de dimensionalidad) y dos técnicas de **clustering** (KMeans, DBSCAN) sobre lecturas de
sensores de monitoreo estructural (SHM), para condensar 5 señales redundantes en componentes interpretables y explorar
si los estados de daño de una estructura emergen como agrupaciones naturales — sin usar la etiqueta de condición para
calcularlas.

## Qué recorre este notebook
1. Carga y limpieza del dataset de sensores
2. Exploración: estadísticas descriptivas, distribución de clases, correlación entre sensores
3. Estandarización y PCA (varianza explicada, proyección 2D)
4. Clustering no supervisado: KMeans (método del codo) y DBSCAN (densidad)
5. Comparación de ambos métodos contra la etiqueta real (Silhouette, ARI)
6. Loadings de PCA y una comparación rápida de clasificación (features originales vs. componentes PCA)


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, silhouette_score, adjusted_rand_score

%matplotlib inline
sns.set_theme(style='whitegrid')
print('✅ Entorno listo (labs/.venv).')


## Contexto del dataset (Kaggle SHM)

| Variable | Unidad | En obra significa… |
|----------|--------|-------------------|
| Accel_X, Accel_Y, Accel_Z | m/s² | Vibración y movimiento en tres ejes |
| Strain | με | Deformación del material (extensómetro) |
| Temp | °C | Temperatura ambiente / del sensor |
| **Condition Label** | **0 / 1 / 2** | **Estado estructural (normal / daño menor / severo)** |

Fuente: Ziya07 · 1 000 lecturas · PCA **no supervisado** (no usa la etiqueta para crear componentes).

Detalle ampliado: [`data/DATOS.md`](data/DATOS.md).


## 1. PCA en monitoreo estructural

En obra, decenas de sensores generan datos redundantes; PCA condensa esa información en pocas componentes
interpretables. Es un método **no supervisado** — solo usa las features de sensores, nunca `Condition Label` — lo
que lo hace útil para reducir ruido y redundancia (por ejemplo, ejes de aceleración correlacionados) antes de
visualizar estados o acelerar un clasificador aguas abajo.

In [ ]:
METODO_REDUCCION = "pca"
print(f"Método elegido: {METODO_REDUCCION}")


## 2. Carga del dataset de sensores

El CSV trae 5 features de sensor + `Timestamp` + `Condition Label` (7 columnas en total). `Timestamp` ordena las
series temporales pero no entra al PCA — solo las 5 lecturas numéricas de sensor.

In [ ]:
# --- Carga desde data/building_health_monitoring_dataset.csv ---
RUTA_DATOS = Path("data/building_health_monitoring_dataset.csv")
df = pd.read_csv(RUTA_DATOS)
print(f"Archivo: {RUTA_DATOS} | Forma: {df.shape[0]} filas × {df.shape[1]} columnas")

FEATURES = [
    "Accel_X (m/s^2)",
    "Accel_Y (m/s^2)",
    "Accel_Z (m/s^2)",
    "Strain (με)",
    "Temp (°C)",
]
N_FILAS_HEAD = 5
print(f"Features para PCA: {FEATURES}")
print(f"Primeras {N_FILAS_HEAD} lecturas:")
display(df.head(N_FILAS_HEAD))


## 3. Calidad de datos y limpieza

Antes de cualquier análisis, revisamos nulos por sensor y descartamos filas incompletas. `Strain` es el sensor más
crítico a inspeccionar primero por su sensibilidad directa al daño estructural.

In [ ]:
# --- Conteo de nulos y limpieza ---
n_antes = len(df)
n_nulos_por_col = df[FEATURES].isna().sum()
print("Nulos por sensor (datos crudos):")
display(n_nulos_por_col)

df_limpio = df.dropna(subset=FEATURES).copy()
n_despues = len(df_limpio)
print(f"Tras dropna: {n_antes} → {n_despues} lecturas válidas")

COLUMNA_REVISAR = "Strain (με)"
stats_col = df[COLUMNA_REVISAR].describe()
print(f"Estadísticas de «{COLUMNA_REVISAR}» (datos crudos):")
display(stats_col)


## 4. Estadísticas descriptivas de sensores

Los sensores tienen magnitudes muy distintas (`Accel_Z` ronda 9.8 m/s² por la gravedad, mientras `Strain` ronda 80
με) — esa dispersión relativa (std vs. |media|) es exactamente lo que exige escalar los datos antes de aplicar
PCA, o una feature de gran magnitud dominaría artificialmente la primera componente.

In [ ]:
COLUMNAS_RESUMEN = ["Strain (με)", "Temp (°C)", "Accel_Z (m/s^2)"]
resumen = df_limpio[COLUMNAS_RESUMEN].describe()
display(resumen)
medias_abs = df_limpio[COLUMNAS_RESUMEN].abs().mean()
dispersion = (df_limpio[COLUMNAS_RESUMEN].std() / medias_abs).sort_values(ascending=False)
display(dispersion)


## 5. Distribución de Condition Label

Es típico en SHM real que la clase mayoritaria sea "normal" (0): la estructura pasa la mayor parte del tiempo en
servicio sin daño, y los estados 1 (daño menor) y 2 (daño severo) son, por diseño, minoritarios.

In [ ]:
# --- Histograma de etiquetas ---
conteo = df_limpio['Condition Label'].value_counts().sort_index().to_dict()
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(conteo.keys(), conteo.values(), color=['#2ecc71', '#f39c12', '#e74c3c'], edgecolor='white')
ax.set_xlabel('Condition Label')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribución de estados estructurales')
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['0 — Normal', '1 — Daño menor', '2 — Daño severo'])
plt.tight_layout()
plt.show()

N_CLASES_MOSTRAR = 3
serie_clases = pd.Series(conteo).sort_index().head(N_CLASES_MOSTRAR)
display(serie_clases)


## 6. Correlación entre sensores

Una correlación alta entre dos sensores (por ejemplo, entre ejes de aceleración) es exactamente el tipo de
redundancia que PCA está diseñado para condensar en menos dimensiones.

In [ ]:
# --- Matriz de correlación ---
corr = df_limpio[FEATURES].corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Correlación entre sensores (datos limpios)')
plt.tight_layout()
plt.show()

TOP_N_PARES = 3
pares = []
for i, c1 in enumerate(FEATURES):
    for c2 in FEATURES[i + 1:]:
        pares.append((c1, c2, abs(corr.loc[c1, c2])))
pares_ordenados = sorted(pares, key=lambda x: x[2], reverse=True)
top_pares = pares_ordenados[:TOP_N_PARES]
max_corr = top_pares[0][2]
top_par = (top_pares[0][0], top_pares[0][1])
for c1, c2, r_val in top_pares:
    print(f"  {c1} ↔ {c2}: |r| = {r_val:.3f}")


## 7. Estandarización antes de PCA

`StandardScaler` es recomendable antes de PCA precisamente porque, sin escalar, un sensor de mayor magnitud absoluta
(aquí `Accel_Z`, dominado por la gravedad) se roba la primera componente sin que eso refleje realmente más
información estructural.

In [ ]:
# --- Comparar varianza de PC1 con y sin escalado ---
X_raw = df_limpio[FEATURES].values
scaler_ref = StandardScaler()
X_scaled_ref = scaler_ref.fit_transform(X_raw)

pca_crudo = PCA(n_components=5, random_state=42)
pca_crudo.fit(X_raw)
var_pc1_crudo = float(pca_crudo.explained_variance_ratio_[0])

pca_esc = PCA(n_components=5, random_state=42)
pca_esc.fit(X_scaled_ref)
var_pc1_escalado = float(pca_esc.explained_variance_ratio_[0])

print(f"PC1 sin escalado:  {var_pc1_crudo*100:.1f}% de varianza")
print(f"PC1 con escalado:  {var_pc1_escalado*100:.1f}% de varianza")

ESCALAR = True
if ESCALAR:
    scaler = StandardScaler()
    X_pca_input = scaler.fit_transform(X_raw)
else:
    X_pca_input = X_raw
pca_tmp = PCA(n_components=5, random_state=42)
pca_tmp.fit(X_pca_input)
var_pc1_actual = float(pca_tmp.explained_variance_ratio_[0])
print(f"ESCALAR={ESCALAR} → PC1 explica {var_pc1_actual*100:.1f}% de varianza")


## 8. Varianza explicada (scree plot)

Con solo 5 sensores de entrada, capturar el 90% de la varianza típicamente requiere casi todas las componentes —
el "codo" de compresión es modesto en este dataset, pero el ejercicio de mirarlo es el mismo que harías con
decenas de sensores reales, donde la reducción sí es drástica.

In [ ]:
# --- PCA completo y scree plot ---
pca_full = PCA(random_state=42)
pca_full.fit(X_pca_input)
var_ratio = pca_full.explained_variance_ratio_
var_acum = np.cumsum(var_ratio).tolist()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(1, len(var_ratio) + 1), var_ratio, color='#3498db', edgecolor='white')
axes[0].set_xlabel('Componente principal')
axes[0].set_ylabel('Varianza explicada')
axes[0].set_title('Scree plot — varianza por componente')

axes[1].plot(range(1, len(var_acum) + 1), var_acum, 'o-', color='#e74c3c')
axes[1].axhline(0.90, color='gray', linestyle='--', label='90%')
axes[1].set_xlabel('Número de componentes')
axes[1].set_ylabel('Varianza acumulada')
axes[1].set_title('Varianza acumulada')
axes[1].legend()
plt.tight_layout()
plt.show()

UMBRAL_VARIANZA = 0.90
N_COMPONENTES = 5
n_min = int(np.searchsorted(var_acum, UMBRAL_VARIANZA) + 1)
print(f"Umbral {UMBRAL_VARIANZA:.0%} → mínimo {n_min} componente(s)")
print(f"N_COMPONENTES = {N_COMPONENTES}")


## 9. Proyección 2D: PC1 vs PC2

Proyectamos las lecturas sobre el plano PC1–PC2 y coloreamos por `Condition Label` — pero solo para **interpretar**
el gráfico después, no porque PCA haya usado la etiqueta al calcular las componentes.

In [ ]:
# --- Transformación a componentes principales ---
X_pca = pca_full.transform(X_pca_input)
var_pc1 = float(var_ratio[0])
var_pc2 = float(var_ratio[1])
print(f"Proyección lista: {X_pca.shape[0]} puntos × {X_pca.shape[1]} componentes")

COLOREAR_POR = "Condition Label"
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(
    X_pca[:, 0], X_pca[:, 1],
    c=df_limpio[COLOREAR_POR], cmap='RdYlGn_r', alpha=0.7, edgecolors='white', linewidth=0.3,
)
ax.set_xlabel(f'PC1 ({var_pc1*100:.1f}% varianza)')
ax.set_ylabel(f'PC2 ({var_pc2*100:.1f}% varianza)')
ax.set_title('Proyección PCA — estados estructurales')
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label(COLOREAR_POR)
plt.tight_layout()
plt.show()
n_muestras = len(X_pca)


## 10. KMeans y método del codo

KMeans agrupa las lecturas de sensores **sin usar** `Condition Label`; más adelante comparamos esos clústeres
contra la etiqueta real. El gráfico del codo sugiere k≈3 — consistente con los tres estados de daño esperados — y
un Silhouette moderado es normal en datos SHM reales, donde los límites entre estados no son perfectamente nítidos.

In [ ]:
# --- Inercia vs k (espacio escalado) ---
K_MIN, K_MAX = 2, 10
inertias = []
for k in range(K_MIN, K_MAX + 1):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_pca_input)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(K_MIN, K_MAX + 1), inertias, 'o-', color='#2980b9', linewidth=2)
ax.set_xlabel('Número de clústeres k')
ax.set_ylabel('Inercia (within-cluster SS)')
ax.set_title('Método del codo — KMeans sobre sensores escalados')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

K_OPT = 3
kmeans = KMeans(n_clusters=K_OPT, random_state=42, n_init=10)
labels_km = kmeans.fit_predict(X_pca_input)
sil_km = silhouette_score(X_pca_input, labels_km)
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels_km, cmap='tab10', alpha=0.7)
ax.set_xlabel(f'PC1 ({var_pc1*100:.1f}% varianza)')
ax.set_ylabel(f'PC2 ({var_pc2*100:.1f}% varianza)')
ax.set_title('KMeans k=3 — agrupación en plano PCA')
plt.colorbar(sc, ax=ax, label='Clúster')
plt.tight_layout()
plt.show()
print(f"Silhouette KMeans = {sil_km:.3f}")


## 11. DBSCAN (densidad y ruido)

A diferencia de KMeans, DBSCAN no exige fijar **k** de antemano: detecta regiones densas y marca como ruido
(etiqueta **-1**) las lecturas atípicas de sensores. `eps` pequeño produce más ruido; `eps` grande colapsa todo en
pocos clústeres — vale la pena mirar esa sensibilidad antes de fijar un valor.

In [ ]:
# --- Sensibilidad rápida a eps ---
for eps_demo in (0.5, 0.7, 1.0):
    db_demo = DBSCAN(eps=eps_demo, min_samples=8).fit(X_pca_input)
    n_cl = len(set(db_demo.labels_)) - (1 if -1 in db_demo.labels_ else 0)
    n_noise = int((db_demo.labels_ == -1).sum())
    print(f'eps={eps_demo:.1f} → {n_cl} clúster(es), {n_noise} puntos ruido (-1)')

EPS = 0.7
MIN_SAMPLES = 8
dbscan = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES)
labels_db = dbscan.fit_predict(X_pca_input)
n_clusters_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise_db = int((labels_db == -1).sum())
mask_db = labels_db != -1
sil_db = silhouette_score(X_pca_input[mask_db], labels_db[mask_db])
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels_db, cmap='coolwarm', alpha=0.7)
ax.set_title('DBSCAN — clústeres y puntos ruido')
plt.colorbar(sc, ax=ax, label='Clúster')
plt.tight_layout()
plt.show()
print(f"Clústeres={n_clusters_db}, ruido={n_noise_db}, silhouette={sil_db:.3f}")


## 12. Comparativa KMeans vs DBSCAN vs etiqueta real

El índice de **Silhouette** mide qué tan bien separados están los clústeres (más alto = mejor), sin mirar ninguna
etiqueta externa. El **Adjusted Rand Index (ARI)** sí compara los clústeres encontrados contra `Condition Label` —
solo para evaluación, nunca para calcular los clústeres. Con estos parámetros, KMeans suele alinearse mejor que
DBSCAN con los estados estructurales conocidos.

In [ ]:
# --- Etiqueta de referencia (supervisada, solo para comparar) ---
y_true = df_limpio['Condition Label'].values
ari_km = adjusted_rand_score(y_true, labels_km)
ari_db = adjusted_rand_score(y_true[mask_db], labels_db[mask_db]) if mask_db.sum() > 0 else float('nan')
print("Etiquetas reales: 0=normal, 1=daño menor, 2=severo")

METRICA_PRIORITARIA = 'ari'
comparativa = pd.DataFrame({
    'Método': ['KMeans', 'DBSCAN'],
    'k / clústeres': [K_OPT, n_clusters_db],
    'Silhouette': [sil_km, sil_db],
    'ARI vs Condition Label': [ari_km, ari_db],
    'Puntos ruido (-1)': [0, n_noise_db],
})
display(comparativa.round(3))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x = np.arange(2)
axes[0].bar(x, comparativa['Silhouette'], color=['#3498db', '#e67e22'])
axes[0].set_xticks(x); axes[0].set_xticklabels(comparativa['Método'])
axes[0].set_title('Silhouette')
axes[1].bar(x, comparativa['ARI vs Condition Label'], color=['#3498db', '#e67e22'])
axes[1].set_xticks(x); axes[1].set_xticklabels(comparativa['Método'])
axes[1].set_title('ARI vs daño real')
plt.tight_layout()
plt.show()


## 13. Loadings y clasificación (original vs PCA)

Los **loadings** de PC1 muestran qué sensor pesa más en la primera componente — típicamente `Strain`, coherente
con que la deformación es la señal más directa de daño estructural. Cerramos comparando qué tanto accuracy se
pierde (si es que se pierde algo) al clasificar con solo 3 componentes PCA en vez de las 5 features originales.

In [ ]:
# --- Matriz de loadings ---
loadings = pd.DataFrame(
    pca_full.components_.T,
    index=FEATURES,
    columns=[f'PC{i+1}' for i in range(len(FEATURES))],
)
feature_pc1 = loadings['PC1'].abs().idxmax()

fig, ax = plt.subplots(figsize=(8, 4))
loadings['PC1'].sort_values().plot(kind='barh', ax=ax, color='#9b59b6')
ax.set_title(f'Loadings PC1 — mayor peso: {feature_pc1}')
ax.set_xlabel('Peso en PC1')
plt.tight_layout()
plt.show()
display(loadings.round(3))

N_COMPONENTES_ML = 3
TEST_SIZE = 0.2
RANDOM_STATE = 42
X_ml = StandardScaler().fit_transform(df_limpio[FEATURES])
y_ml = df_limpio['Condition Label']
X_train, X_test, y_train, y_test = train_test_split(
    X_ml, y_ml, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_ml,
)
rf_orig = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
rf_orig.fit(X_train, y_train)
acc_orig = accuracy_score(y_test, rf_orig.predict(X_test))
pca_ml = PCA(n_components=N_COMPONENTES_ML, random_state=RANDOM_STATE)
X_train_pca = pca_ml.fit_transform(X_train)
X_test_pca = pca_ml.transform(X_test)
rf_pca = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
rf_pca.fit(X_train_pca, y_train)
acc_pca = accuracy_score(y_test, rf_pca.predict(X_test_pca))
print(f"Accuracy original: {acc_orig:.3f} | PCA-3: {acc_pca:.3f}")


## Reflexión: preguntas que los alumnos necesitarían

- ¿Cuántas componentes principales son "suficientes" para una decisión de ingeniería, y quién decide ese umbral —
  un criterio estadístico (90% de varianza) o el costo de instrumentar/monitorear menos señales?
- Si un clúster de KMeans no coincide con `Condition Label` para un edificio real, ¿qué harías primero: dudar del
  sensor, dudar del modelo, o dudar de la etiqueta histórica?
- DBSCAN marcó ciertos puntos como "ruido" (-1). ¿Bajo qué condiciones ese ruido sería en realidad la señal más
  importante (por ejemplo, el inicio de un evento sísmico o un sensor fallando)?
- PCA es no supervisado y nunca vio `Condition Label`. ¿Qué riesgo hay en interpretar visualmente "esta zona del
  plano PC1–PC2 es peligrosa" y actuar sobre eso sin una validación adicional?
- Si tuvieras 50 sensores en vez de 5, ¿cómo cambiaría tu criterio para decidir cuántas componentes conservar?
